In [1]:
import os
import time
import folium
import tomllib
import shapely
import sqlparse
import warnings
import pymysql
import numpy as np
import pandas as pd
import datetime as dt
from tqdm import tqdm
from typing import Union
from pathlib import Path
from dotenv import dotenv_values
from shapely.geometry import Polygon
from shapely import points, contains, prepare

# Setting Config File

As we have seen in previous notebooks, it is possible to make all the checks for the seismic data using similar vectorization techniques (such as polygon, numeric, etc). However, define and work around each one of the possible checks can be annoying and time-consuming. For this reason, we will define a configuration file where we can set all the checks that we want to perform on the seismic data. This way, we can easily modify the checks without having to change the code. We will use a TOML file for this purpose, as it is a simple and human-readable format.

The main idea here is to design a new scheme using a config file (in format TOML) that allows to define the checks that we want to perform on the seismic data. To achieve this, we will define a TOML file and a wrapper function that reads the config file and executes the checks accordingly. This way, we can easily modify the checks without having to change the code.

## Configuration for query data from seiscomp

The first step is to define the configuration for querying data from Seiscomp. This configuration will include the connection parameters for the database, as well as the query that we want to execute to retrieve the seismic data. We will define this configuration in a TOML file, which will be read by our code to establish the connection and execute the query. The configuration will require the following parameters:

- host: The hostname of the database server.
- port: The port number of the database server.
- user: The username to connect to the database.
- password: The password to connect to the database.
- database: The name of the database to connect to.
- query: The SQL query to execute to retrieve the seismic data.

The recommended way to store the database credentials is to use a .env file, which will be read by our code to set the environment variables. This way, we can keep the credentials secure and avoid hardcoding them in the code or the config file. Moreover, the SQL query can be defined in a separate .sql file, which will be read by our code to execute the query. This way, we can easily modify the query without having to change the code or the config file. The TOML then will reference the .sql file to read the query. This approach allows for a clean separation of concerns and makes it easier to manage and update the configuration for querying data from Seiscomp.

The structure of the TOML file for the Seiscomp configuration will look like this:
```toml
[database]
env_file = ".env"
# Not required if env_file is provided and contains the host variable
host = "localhost"
port = 3306
user = "username"
password = "password"
database = "seiscomp"

[query]
sql_file = "query.sql"
```

If .env file is provided, it must have the following variables:
```
# Credentials for the database connection
DB_HOST=localhost
DB_PORT=3306
DB_USER=username
DB_PASSWORD=password
DB_NAME=seiscomp

# Credentials for any other database connection parameters that might be needed too
DB_HOST2=localhost
DB_PORT2=3306
DB_USER2=username
DB_PASSWORD2=password
DB_NAME2=seiscomp
```

The advantage of using this approach is that we can easily modify the database connection parameters and the SQL query without having to change the code. We can simply update the .sql file or the .env file (adding more environment variables if needed) and the code will read the new configuration and execute the checks accordingly. This makes it easier to manage and maintain the code, as well as to adapt it to different environments or requirements.

In [4]:
def load_config(
        config_path: str,
        credentials_keys: tuple[str, str, str, str, str | int] = ("host", "user", "password", "database", 3306)
) -> dict:
    """
    Read and validate a TOML configuration file for the seismic revision
    database connection and query setup. It resolves all external references
    (env file, SQL file) and returns a single, self-contained config dict ready
    for use.

    Parameters
    ----------
    config_path : str
        Path to the .toml configuration file.

    credentials_keys: tuple[str]
        List of credentials keys for special cases (HOST, USER, PWD, DB, PORT). Defaults to ("host", "user", "password", "database", 3306)

    Returns
    -------
    dict with keys:
        "credentials" : dict   – resolved DB connection parameters
        "sql"         : str    – SQL query string loaded from the .sql file

    Raises
    ------
    FileNotFoundError
        If the TOML file, the .env file (when required), or the .sql file
        cannot be found.
    KeyError
        If a required credential is missing from both the env file and the
        TOML [database] section.
    ValueError
        If the [database] or [query] sections are missing from the TOML,
        or if the SQL file path is not declared.
    TypeError
        If port is declared but cannot be interpreted as an integer.
    """
    # ── 1. Load the TOML file ──────────────────────────────────────
    config_path = Path(config_path).resolve()
    if not config_path.exists():
        raise FileNotFoundError(f"Configuration file not found: {config_path}")
    if config_path.suffix.lower() != ".toml":
        warnings.warn(
            f"Expected a .toml file but got {config_path.suffix!r}. "
            "Attempting to parse anyway.",
            stacklevel=2
        )

    with open(config_path, "rb") as fh:
        raw = tomllib.load(fh)

    # Base directory for resolving relative paths declared inside the TOML
    base_dir = config_path.parent

    # ── 2. Validate top-level sections ────────────────────────────
    if "database" not in raw:
        raise ValueError(
            f"Missing [database] section in {config_path.name}. "
            "Please declare connection parameters or an env_file path."
        )
    if "query" not in raw:
        raise ValueError(
            f"Missing [query] section in {config_path.name}. "
            "Please declare 'sql_file'."
        )

    db_cfg    = raw["database"]
    query_cfg = raw["query"]

    env_values: dict[str, str] = {}
    env_file_path = db_cfg.get("env_file")

    if env_file_path is not None:
        env_file_path = Path(env_file_path)
        # Resolve relative paths against the TOML's own directory
        if not env_file_path.is_absolute():
            env_file_path = (base_dir / env_file_path).resolve()

        if not env_file_path.exists():
            raise FileNotFoundError(
                f"env_file declared in {config_path.name} was not found: "
                f"{env_file_path}\n"
                "Either create the file, fix the path, or remove the "
                "'env_file' key and supply credentials directly in [database]."
            )

        env_values = dotenv_values(env_file_path)  # does NOT mutate os.environ

        if not env_values:
            warnings.warn(
                f"The env_file at {env_file_path} was found but appears to be "
                "empty. Falling back to inline TOML credentials.",
                stacklevel=2
            )
    else:
        warnings.warn(
            "No 'env_file' declared in [database]. "
            "Reading credentials from the TOML file directly. "
            "Avoid committing plaintext passwords to version control.",
            stacklevel=2
        )

    def _resolve(key: str | int, *, required: bool = True, default=None):
        """
        Resolve a single credential using the priority chain:
        os.environ > env_file > TOML inline > default.

        Raises KeyError for required keys that cannot be resolved anywhere.
        """
        # 1. Live environment variable (uppercase key by convention)
        value = os.environ.get(key.upper()) or os.environ.get(key)
        if value is not None:
            return value

        # 2. .env file
        value = env_values.get(key.upper()) or env_values.get(key)
        if value is not None:
            return value

        # 3. Inline TOML
        value = db_cfg.get(key)
        if value is not None:
            return value

        # 4. Default / missing
        if required:
            sources = []
            if env_file_path:
                sources.append(f"env_file ({env_file_path.name})")
            sources.append(f"[database] in {config_path.name}")
            sources.append("os.environ")
            raise KeyError(
                f"Required credential {key!r} was not found in any of: "
                + ", ".join(sources)
            )
        return default

    # Credential keys should return strings except for port:
    names = ('HOST', 'USER', 'PWD', 'DB')
    for i in range(0, len(credentials_keys) - 1):
        if not isinstance(credentials_keys[i], str):
            raise TypeError(
                f"{names[i]} must be an string, got {credentials_keys[i]!r} (type: {type(credentials_keys[i]).__name__})"
            )

    # Resolve each credential individually so errors are precise
    host     = _resolve(credentials_keys[0], required=True)
    user     = _resolve(credentials_keys[1], required=True)
    password = _resolve(credentials_keys[2], required=True)
    database = _resolve(credentials_keys[3], required=True)
    port_raw = _resolve(credentials_keys[4], required=False, default=3306)

    # Port coercion — env files and os.environ deliver strings, TOML delivers int
    try:
        port = int(port_raw)
    except (TypeError, ValueError):
        raise TypeError(
            f"'port' must be an integer, got {port_raw!r} "
            f"(type: {type(port_raw).__name__}). "
            "Check the value in your env_file or [database] section."
        )

    if not (1 <= port <= 65535):
        raise ValueError(
            f"'port' value {port} is outside the valid range 1–65535."
        )

    credentials = {
        "host":     host,
        "port":     port,
        "user":     user,
        "password": password,
        "database": database,
    }

    # ── 4. Load SQL query ──────────────────────────────────────────
    sql_file_key = query_cfg.get("sql_file")
    if not sql_file_key:
        raise ValueError(
            f"Missing 'sql_file' key in [query] section of {config_path.name}. "
            "Declare the path to the .sql file to use."
        )

    sql_path = Path(sql_file_key)
    if not sql_path.is_absolute():
        sql_path = (base_dir / sql_path).resolve()

    if not sql_path.exists():
        raise FileNotFoundError(
            f"SQL file declared in {config_path.name} was not found: {sql_path}\n"
            "Check the 'sql_file' path under [query]."
        )

    if sql_path.suffix.lower() != ".sql":
        warnings.warn(
            f"Expected a .sql file but got {sql_path.suffix!r}. "
            "Attempting to read anyway.",
            stacklevel=2
        )

    sql_text = sql_path.read_text(encoding="utf-8").strip()

    if not sql_text:
        warnings.warn(
            f"The SQL file at {sql_path} is empty. "
            "Queries will fail at execution time.",
            stacklevel=2
        )

    # ── 5. Return resolved configuration ──────────────────────────
    return {
        "credentials": credentials,
        "sql":         sql_text,
    }

In [9]:
# Test the config loader with a sample TOML file
if __name__ == "__main__":
    try:
        config = load_config("config.toml", credentials_keys=('SERVER_SC6_HOST', 'SERVER_SC6_USERNAME', 'SERVER_SC6_PASSWORD', 'SERVER_SC6_DATABASE', 'SERVER_SC6_PORT'))
        print("Configuration loaded successfully")
    except Exception as e:
        print(f"Error loading configuration: {e}")

Configuration loaded successfully


## Vectorization functions

As we stated in the Vectorization notebook, we can define different vectorization functions that will be used to perform the checks on the seismic data. In general, we have:

In [2]:
# Numeric comparisons
def numeric_mask(
    events: pd.DataFrame,
    column: str,
    mode: str,
    threshold : float | None= None,
    lower: float | None = None,
    upper: float | None = None,
    dtype = np.float64
) -> np.ndarray:
    """
    Vectorized numeric comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    column : str
        Column to evaluate.
    mode : str
        Comparison mode:
            'gt'       -> values > threshold
            'ge'       -> values >= threshold
            'lt'       -> values < threshold
            'le'       -> values <= threshold
            'eq'       -> values == threshold
            'ne'       -> values != threshold
            'between'  -> lower <= values <= upper
            'outside'  -> values < lower or values > upper
            'abs_gt'   -> abs(values) > threshold
            'abs_ge'   -> abs(values) >= threshold
    threshold : float, optional
        Threshold for one-sided and equality comparisons.
    lower, upper : float, optional
        Bounds for range comparisons.
    dtype : numpy dtype
        Target dtype for NumPy conversion.

    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    values = events[column].to_numpy(dtype=dtype, copy=False)

    if mode == 'gt':
        return values > threshold
    elif mode == 'ge':
        return values >= threshold
    elif mode == 'lt':
        return values < threshold
    elif mode == 'le':
        return values <= threshold
    elif mode == 'eq':
        return values == threshold
    elif mode == 'ne':
        return values != threshold
    elif mode == 'between':
        return (values >= lower) & (values <= upper)
    elif mode == 'outside':
        return (values < lower) | (values > upper)
    elif mode == 'abs_gt':
        return np.abs(values) > threshold
    elif mode == 'abs_ge':
        return np.abs(values) >= threshold
    else:
        raise ValueError(f"Unsupported mode: {mode!r}")

In [3]:
# Column vs column comparisons
def column_column_mask(
    events: pd.DataFrame,
    left_col: str,
    mode: str,
    right_col: str,
    offset: float = 0.0,
    factor: float = 1.0,
    dtype=np.float64,
) -> np.ndarray:
    """
    Vectorized column to column comparator for seismic quality checks.
    It follows: events[left_col] <<mode>> factor * events[right_col] + offset

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    left_col : str
        Left column to compare.
    mode : str
        Comparison mode:
            'gt'       -> values > threshold
            'ge'       -> values >= threshold
            'lt'       -> values < threshold
            'le'       -> values <= threshold
            'eq'       -> values == threshold
            'ne'       -> values != threshold
    right_col : str
        Right column to compare.
    offset : float, optional
        Offset of the equation, if required. Defaults to zero.
    factor : float, optional
        Multiplier for the right column, if required. Defaults to zero.
    dtype : numpy dtype
        Target dtype for NumPy conversion.
    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    left = events[left_col].to_numpy(dtype=dtype, copy=False)
    right = events[right_col].to_numpy(dtype=dtype, copy=False) * factor + offset
    ops = {
        "gt": np.greater,
        "ge": np.greater_equal,
        "lt": np.less,
        "le": np.less_equal,
        "eq": np.equal,
        "ne": np.not_equal,
    }
    return ops[mode](left, right)

In [4]:
# Non-numeric comparisons
def non_numeric_mask(
    events: pd.DataFrame,
    column: str,
    mode: str,
    values: list[str] | None = None,
) -> np.ndarray:
    """
    Vectorized non-numeric comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    column : str
        Column to evaluate.
    mode : str
        Comparison mode:
            'is_null'       -> values that are null/NaN
            'not_null'      -> values that are not null/NaN
            'in'            -> values that are in the provided list
            'not_in'        -> values that are not in the provided list
    values : list[str], optional
        List of values for 'in' or 'not_in' modes.
    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    s = events[column]
    if mode == "is_null":
        return s.isna().to_numpy()
    elif mode == "not_null":
        return s.notna().to_numpy()
    elif mode == "in":
        return s.isin(values).to_numpy()
    elif mode == "not_in":
        return (~s.isin(values)).to_numpy()
    else:
        raise ValueError(f"Unsupported category mode: {mode}")

In [5]:
# Polygonal comparison
def build_polygon_mask(
    events: pd.DataFrame,
    lon_col: str,
    lat_col: str,
    polygon: Union[Polygon, shapely.geometry.base.BaseGeometry],
    mode : str = "inside"
) -> np.ndarray:
    """
    Vectorized polygon comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    lon_col : str
        Longitude column to evaluate.
    lat_col : str
        Latitude column to evaluate.
    polygon : shapely.geometry.Polygon or shapely.geometry.base.BaseGeometry
        Shapely polygon to compare.
    mode : str
        Comparison mode:
            'inside'       -> values inside polygon
            'outside'      -> values outside polygon
    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    if mode not in ['inside', 'inside']:
        raise ValueError(f"Unsupported polygon mode: {mode}. Accepted modes: 'inside' and 'outside'")
    if not shapely.is_prepared(polygon):
        shapely.prepare(polygon)
    inside = shapely.contains_xy(
        polygon,
        events[lon_col].to_numpy(),
        events[lat_col].to_numpy()
    )
    return inside if mode == "inside" else ~inside

In [6]:
# Temporal comparison
def temporal_mask(
    events: pd.DataFrame,
    column: str,
    mode: str,
    value: str,
) -> np.ndarray:
    """
    Vectorized temporal comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    column : str
        Column to evaluate.
    mode : str
        Comparison mode:
            'gt'       -> values > Timestamp
            'ge'       -> values >= Timestamp
            'lt'       -> values < Timestamp
            'le'       -> values <= Timestamp
            'eq'       -> values == Timestamp
            'ne'       -> values != Timestamp
    value : str
        Reference timestamp used for the comparison. Any string accepted by
        :class:`pandas.Timestamp` can be supplied, for example
        '2024-01-01T00:00:00Z' or '2024-01-01 00:00:00'.

    Returns
    -------
    numpy.ndarray
        Boolean mask with one entry per row in events. True indicates
        that the row satisfies the requested temporal condition.
    """
    left = pd.to_datetime(events[column], utc=False)
    right = pd.Timestamp(value)

    ops = {
        "gt": np.greater,
        "ge": np.greater_equal,
        "lt": np.less,
        "le": np.less_equal,
        "eq": np.equal,
        "ne": np.not_equal,
    }
    return ops[mode](left.to_numpy(), right.to_datetime64())

In [7]:
# Composed rules
def combine_masks(
        masks: list[np.ndarray],
        logic: str = "and"
) -> np.ndarray:
    """
    Combine multiple boolean masks using a logical operator.

    Parameters
    ----------
    masks : list[np.ndarray]
        List of boolean masks to combine. All masks must have the same shape.
    logic : str
        Combination logic to apply:
            'and' -> logical AND across all masks
            'or'  -> logical OR across all masks
            'xor' -> logical XOR between exactly 2 masks

    Returns
    -------
    np.ndarray
        Boolean mask with one entry per element in the input masks.

    Raises
    ------
    ValueError
        If no masks are provided, if 'xor' is used with anything other than
        exactly 2 masks, or if an unsupported logic value is supplied.
    """
    if not masks:
        raise ValueError("No masks provided")
    if logic == "and":
        return np.logical_and.reduce(masks)
    elif logic == "or":
        return np.logical_or.reduce(masks)
    elif logic == "xor":
        if len(masks) != 2:
            raise ValueError("XOR logic requires exactly 2 masks")
        return np.logical_xor(masks[0], masks[1])
    else:
        raise ValueError(f"Unsupported logic: {logic!r}")